# Auswertung von Blutbild-XML-Dateien:

1. Messwerte aus mehreren XML-Dateien einlesen und zu einer Tabelle zusammenführen
2. Referenzwerte aus einer separaten XML-Datei laden
3. Übersichtstabelle mit great_tables erzeugen
4. Zeitlichen Verlauf einzelner Parameter als Graph darstellen



In [61]:
import xml.etree.ElementTree as ET  # Library zum Öffnen und Bearbeiten von XML Dateien
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from great_tables import GT, md, style, loc  # Stellt komplexere und schönere Tabellen her

# Seaborn-Grundstil für alle Plots festlegen
sns.set_style("whitegrid")

# Basisverzeichnis mit den XML-Dateien
DATA_PATH = Path(
    "H:/02 Data Steward Fortbildung/03 Unterrichtsmaterialien/"
    "B IT und Programmiergrundlagen/"
    "B2 Grundlagen von Modellier- und Programmiersprachen/"
    "Abschlussprojekt/04 Sample_Data"
)


# ---------------------------------------------------------------------------
# Funktionen
# ---------------------------------------------------------------------------

In [2]:
def get_xml(path, file_list):
    """
    Liest mehrere Messungs-XML-Dateien ein und führt sie zu einer Tabelle zusammen.

    Rückgabe:
        df_all    : DataFrame (Index = Parameter, Spalten = M1, M2, ... + 'Gruppe')
        label_map : dict {Spaltenname: Markdown-Label mit Datum, Labor, Ort}
        date_map  : dict {Spaltenname: Messdatum als date-Objekt} (für den Plot)
    """
    label_map = {}
    date_map = {}
    df_all = None  # Gesamttabelle, wird pro Datei erweitert

    for idx, file_name in enumerate(file_list):
        col = f"M{idx + 1}"  # Spaltenname der Messung, z. B. "M1"

        tree = ET.parse(Path(path) / f"{file_name}.xml")
        root = tree.getroot()

        # Metadaten und Messwerte dieser Datei sammeln
        lab, place, date = None, None, None
        para_list, ergebnis_list, gruppe_list = [], [], []

        for elem in root:
            elem_class = elem.get("class")

            if elem_class == "Lab":
                # Labor-Name und Ort auslesen
                lab = elem[0].text
                place = elem[1].text

            elif elem_class == "Date":
                # Messdatum auslesen
                date = pd.to_datetime(elem.text, format="%d.%m.%Y")

            elif elem_class == "KlinData":
                # Alle klinischen Parameter der Gruppe (z. B. "Haematologie") auslesen
                for child in elem:
                    para_list.append(child.tag)                # Parametername
                    ergebnis_list.append(float(child[0].text)) # Messwert
                    gruppe_list.append(elem.tag)               # Parametergruppe

        # Label erst nach der Schleife bauen 
        label_map[col] = md(f"{date.strftime('%d.%m.%Y')}<br>{lab}<br>{place}")
        date_map[col] = date.date()

        # Messwerte dieser Datei als DataFrame 
        df_iter = pd.DataFrame({"Parameter": para_list, col: ergebnis_list})
        df_iter = df_iter.set_index("Parameter")

        if df_all is None:
            # Erste Datei: Gruppenspalte mitnehmen
            df_iter["Gruppe"] = gruppe_list
            df_all = df_iter
        else:
            # Weitere Dateien: über den Parameter-Index anfügen
            df_all = df_all.merge(df_iter, left_index=True,
                                  right_index=True, how="outer")

    return df_all, label_map, date_map


In [3]:
def get_ref(path, filename):
    """
    Liest die Referenzwerte-XML ein.

    Erwartete Kind-Reihenfolge je Parameter:
        child[0] = Obergrenze, child[1] = Untergrenze, child[2] = Einheit
    """
    tree = ET.parse(Path(path) / filename)
    root = tree.getroot()

    para_list, upper_list, lower_list, referenz_list, einheit_list = [], [], [], [], []

    for elem in root:
        for child in elem:
            para_list.append(child.tag)
            upper_list.append(float(child[0].text))   # Obergrenze
            lower_list.append(float(child[1].text))   # Untergrenze
            referenz_list.append(f"{child[1].text} - {child[0].text}")
            einheit_list.append(child[2].text)

    ref = pd.DataFrame({
        "Parameter":   para_list,
        "Untergrenze": lower_list,
        "Obergrenze":  upper_list,
        "Referenz":    referenz_list,
        "Einheit":     einheit_list,
    })
    return ref



In [4]:
def plot_parameter(df, date_map, ref, parameter):
    """
    Stellt den zeitlichen Verlauf eines Parameters dar,
    inkl. grün markiertem Referenzbereich.
    """
    # Gruppenspalte entfernen und transponieren:
    # Zeilen = Messungen, Spalten = Parameter
    df_t = df.drop("Gruppe", axis=1).transpose()
    df_t["Datum"] = [date_map[col] for col in df_t.index]

    # Verlaufslinie zeichnen
    ax = sns.lineplot(data=df_t, x="Datum", y=parameter, marker="o")

    # Referenzbereich einzeichnen
    low = ref.loc[ref["Parameter"] == parameter, "Untergrenze"].iloc[0]
    high = ref.loc[ref["Parameter"] == parameter, "Obergrenze"].iloc[0]
    ax.axhspan(low, high, color="tab:green", alpha=0.12)
    ax.axhline(low, color="tab:green", linestyle="--", linewidth=1)
    ax.axhline(high, color="tab:green", linestyle="--", linewidth=1)

    # X-Achse: nur die tatsächlichen Messdaten als Ticks
    ax.set_xticks(df_t["Datum"])
    plt.tight_layout()
    plt.show()
    return ax

In [5]:
def create_table(data, ref):
    """Verbindet Messdaten mit Referenzbereich und Einheit (über den Parameter-Index)."""
    ref = ref.set_index("Parameter").drop(["Untergrenze", "Obergrenze"], axis=1)
    return data.merge(ref, left_index=True, right_index=True).reset_index()

# ---------------------------------------------------------------------------
# Hauptprogramm
# ---------------------------------------------------------------------------

In [66]:
if __name__ == "__main__":

    # Liste der einzulesenden XML-Dateien (ohne Dateiendung),
    files = ["Blutbild01", "Blutbild02", "Blutbild03", "Blutbild04", "Blutbild05"]

    # XML-Dateien einlesen und in ein DataFrame umwandeln.
    df, label_map, date_map = get_xml(DATA_PATH, files)

    # Referenzwerte (Normbereiche) aus separater XML-Datei laden
    ref = get_ref(DATA_PATH, "ReferenzWerte.xml")

    # Übersichtstabelle mit great_tables (GT) erstellen
    meas_cols = list(label_map.keys())

    tbl = (
        # GT-Objekt aus dem kombinierten DataFrame (Messwerte + Referenzbereiche) erzeugen.
        # rowname_col:   Spalte "Parameter" wird zur Zeilenbeschriftung
        # groupname_col: Spalte "Gruppe" gruppiert die Zeilen inhaltlich
        GT(create_table(df, ref),
           rowname_col="Parameter", groupname_col="Gruppe")

        # Überschrift der Tabelle: Titel + Untertitel
        .tab_header(title="Blutbild", subtitle="Patient: Felix Ostwaldt")

        # Spanner = übergreifende Kopfzeile, die alle Messwert-Spalten
        # unter dem gemeinsamen Label "Messungen" zusammenfasst
        .tab_spanner(label="Messungen", columns=meas_cols)

        # Spaltenüberschriften umbenennen:
        # **label_map entpackt das Dict, sodass jede Messspalte ihr lesbares Label bekommt.
        # "Einheit" und "Referenz" werden explizit (um-)benannt.
        .cols_label(**label_map, Einheit="Einheit", Referenz="Referenzbereich")

        # Alle Messwerte einheitlich mit einer Nachkommastelle formatieren
        .fmt_number(columns=meas_cols, decimals=1)

        # Zahlen rechtsbündig ausrichten 
        .cols_align(columns=meas_cols, align="right")

        # Textspalten (Einheit, Referenzbereich) linksbündig ausrichten
        .cols_align(columns=["Einheit", "Referenz"], align="left")

        .opt_stylize(style=6, color="red")
        
        .tab_style(style=style.fill(color="lightgray"), locations=loc.body(columns=["Einheit", "Referenz"]))
        .tab_style(style=style.fill(color="lightgray"), locations=loc.column_labels(columns=['Einheit', 'Referenz']))
        #.data_color(columns = [-1, -2])
    )
    tbl.show()
    
    #plot_parameter(df, date_map, ref, "Clucose")

Blutbild 
 
 
 Patient: Felix Ostwaldt 
 
 
 
 
 Messungen 
 
 Referenzbereich 
 Einheit 
 
 
 10.05.2025 Labor 24 Berlin 
 20.11.2025 Labor 24 Berlin 
 21.05.2026 My Labor Linz 
 12.08.2026 myLab Linz 
 11.11.2026 My Lab Linz 
 
 
 
 
 Blutbild 
 
 
 Baso.Granulozyten 
 0.1 
 0.1 
 
 0.1 
 0.1 
 0.00 - 0.20 
 G/L 
 
 
 Eosin.Granulozyten 
 0.2 
 0.2 
 
 0.3 
 0.2 
 0.00 - 0.50 
 G/L 
 
 
 Erythrozyten 
 5.1 
 5.1 
 5.1 
 5.0 
 5.1 
 4.4 - 5.9 
 T/L 
 
 
 Hämatokrit 
 46.5 
 45.8 
 45.6 
 45.0 
 42.2 
 40.0 - 52.0 
 % 
 
 
 Hämoglobin 
 16.2 
 15.7 
 15.7 
 15.5 
 14.2 
 13.9 - 17.7 
 g/dL 
 
 
 Leukozyten 
 4.7 
 5.8 
 4.5 
 5.6 
 2.0 
 3.8 - 10.3 
 G/L 
 
 
 Lymphozyten 
 3.5 
 2.0 
 
 1.6 
 3.0 
 1.07 - 4.00 
 G/L 
 
 
 MCH 
 31.5 
 31.0 
 31.0 
 31.2 
 29.4 
 27.3 - 33.5 
 pg 
 
 
 MCHC 
 35.5 
 34.3 
 34.4 
 34.5 
 32.4 
 30.0 - 38.0 
 g/dL 
 
 
 MCV 
 90.1 
 90.3 
 89.9 
 90.4 
 78.8 
 80.0 - 98.0 
 fl 
 
 
 Monozyten 
 0.5 
 0.5 
 
 0.5 
 0.4 
 0.00 - 1.00 
 G/L 
 
 
 RDW 
 12.5 
 
 12.1 
 12.1 
 11.5 
 11.5 - 15.0 
 % 
 
 
 Thrombozyten 
 240.0 
 200.0 
 226.0 
 207.0 
 150.0 
 140 - 400 
 G/L 
 
 
 neutr.Granulozyten 
 7.0 
 3.0 
 
 3.0 
 6.5 
 1.90 - 8.00 
 G/L 
 
 
 KlinChemie 
 
 
 Clucose 
 150.0 
 
 97.0 
 94.0 
 85.0 
 60 - 94 
 mg/dL 
 
 
 Gamma-GT 
 28.0 
 
 26.0 
 34.0 
 15.0 
 0 - 60 
 U/L 
 
 
 HDL-Cholesterin 
 75.0 
 
 69.0 
 75.0 
 102.0 
 40 - 100 
 mg/dL 
 
 
 Harnsäure 
 5.2 
 
 5.1 
 
 4.2 
 3.4 - 7.0 
 mg/dL 
 
 
 Kreatinin 
 0.8 
 
 0.8 
 0.9 
 0.7 
 0 - 1.17 
 mg/dL 
 
 
 LDL-Cholesterin 
 150.0 
 
 93.0 
 77.0 
 70.0 
 0 - 130 
 mg/dL 
 
 
 Non-HDL-Cholesterin 
 110.0 
 
 102.8 
 
 65.0 
 130 - 0 
 mg/dL 
 
 
 Triglyceride 
 55.0 
 
 43.0 
 81.0 
 35.0 
 0 - 130 
 mg/dL 
 
 
 gesCholesterin 
 165.0 
 173.0 
 171.0 
 168.0 
 70.0 
 50 - 200 
 mg/dL 
 
 
 glom.Filtrationsrate 
 120.0 
 
 118.3 
 109.0 
 50.0 
 0 - 90 
 mL/min

# Rückmeldung
Man könnte die Library lxml benutzen